In [ ]:
FLATTEN_FILE_PATH = "/kaggle/input/datasets/hilmawanyusufrukmana/bert-fine-tune-dataset-original-and-flatten-jsonl/flatten_train.jsonl"

# **Data Exploration** 

In [ ]:
import json
import pandas as pd

flat_data = []
with open(FLATTEN_FILE_PATH) as f:
    for line in f:
        flat_data.append(json.loads(line))

flat_df = pd.DataFrame(flat_data)
flat_df.head(5)

In [ ]:
flat_df['article_length'] = flat_df['clean_article'].apply(lambda x: len(x.split()))
flat_df['summary_length'] = flat_df['clean_summary'].apply(lambda x: len(x.split()))

In [ ]:
print("Document length avg:", flat_df['article_length'].mean())
print("Summary length avg:", flat_df['summary_length'].mean())

print("Min document length:", flat_df['article_length'].min())
print("Max document length:", flat_df['article_length'].max())

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.hist(flat_df['article_length'], bins=30)
plt.title("Document length distribution")
plt.xlabel("Word number")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.hist(flat_df['summary_length'], bins=30)
plt.title("Summary length distribution")
plt.xlabel("Word number")
plt.ylabel("Frequency")
plt.show()

In [ ]:
import re

def split_sentences(text: str) -> list:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]

# Ganti bagian loop di script eksplorasi
all_sentence_lengths = []

for article in flat_df['clean_article']:
    sentences = split_sentences(article)
    for sent in sentences:
        all_sentence_lengths.append(len(sent.split()))

df_sent = pd.Series(all_sentence_lengths)

print("> Distribusi Panjang per Kalimat (kata)")
print(f"  Min       : {df_sent.min()}")
print(f"  Max       : {df_sent.max()}")
print(f"  Mean      : {df_sent.mean():.1f}")
print(f"  Median    : {df_sent.median()}")
print(f"  < 3 kata  : {(df_sent < 3).sum():,} ({(df_sent < 3).mean()*100:.1f}%)")
print(f"  > 100 kata: {(df_sent > 100).sum():,} ({(df_sent > 100).mean()*100:.1f}%)")

estimated_max_len = int(df_sent.median() * 2 + 10)
print(f"\n  → Rekomendasi MAX_LEN: {estimated_max_len} token")

# imbalance chechk
total_sentences  = len(all_sentence_lengths)
total_extractive = flat_df['extractive_ids'].apply(lambda x: len(x)).sum()
ratio_pos        = total_extractive / total_sentences * 100

print("\n> Estimasi Class Imbalance")
print(f"  Total kalimat    : {total_sentences:,}")
print(f"  Total extractive : {total_extractive:,}")
print(f"  Estimasi label 1 : {ratio_pos:.1f}%")
print(f"  Estimasi label 0 : {100 - ratio_pos:.1f}%")

# **Splitting**

In [ ]:
!pip install transformers datasets rouge-score -q

In [ ]:
import os

os.makedirs('/kaggle/working/aiindonesia_bootcamp/saved_model', exist_ok=True)

In [ ]:
TRAIN_FILE_PATH = "/kaggle/input/datasets/hilmawanyusufrukmana/bert-fine-tune-dataset-original-and-flatten-jsonl/from_train.jsonl"
TRAINED_MODEL_PATH = "/kaggle/working/aiindonesia_bootcamp/saved_model"

In [ ]:
data = []
with open(TRAIN_FILE_PATH) as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)

print(f"Shape        : {df.shape}")
print(f"Label counts :\n{df['label'].value_counts()}")
df.head(5)

In [ ]:
from sklearn.model_selection import train_test_split

doc_ids = df['doc_id'].unique()

train_ids, temp_ids = train_test_split(doc_ids, test_size=0.2, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)

df_train = df[df['doc_id'].isin(train_ids)].reset_index(drop=True)
df_val = df[df['doc_id'].isin(val_ids)].reset_index(drop=True)
df_test = df[df['doc_id'].isin(test_ids)].reset_index(drop=True)

df_train['text'] = df_train['text'].str.lower()
df_val['text']   = df_val['text'].str.lower()
df_test['text']  = df_test['text'].str.lower()

print(f"Train : {len(df_train):,} kalimat ({len(train_ids):,} dokumen)")
print(f"Val   : {len(df_val):,} kalimat ({len(val_ids):,} dokumen)")
print(f"Test  : {len(df_test):,} kalimat ({len(test_ids):,} dokumen)")


In [ ]:
for name, subset in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    total = len(subset)
    pos   = (subset["label"] == 1).sum()
    neg   = (subset["label"] == 0).sum()
    print(f"{name}")
    print(f"  Label 1 : {pos:,} ({pos/total*100:.1f}%)")
    print(f"  Label 0 : {neg:,} ({neg/total*100:.1f}%)")
    print()

# **Tokenization**

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HUGGINGFACE_TOKEN")

In [ ]:
from huggingface_hub import login
login(token=hf_token)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LEN    = 64
BATCH_SIZE = 32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentenceDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df        = df
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        text  = row["text"]
        label = row["label"]

        encoding = self.tokenizer(
            text,
            max_length     = self.max_len,
            padding        = "max_length",
            truncation     = True,
            return_tensors = "pt"
        )

        return {
            "input_ids"      : encoding["input_ids"].squeeze(0),
            "attention_mask" : encoding["attention_mask"].squeeze(0),
            "label"          : torch.tensor(label, dtype=torch.long)
        }


train_dataset = SentenceDataset(df_train, tokenizer, MAX_LEN)
val_dataset   = SentenceDataset(df_val,   tokenizer, MAX_LEN)
test_dataset  = SentenceDataset(df_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

# Cek satu sampel
sample = train_dataset[0]
print(f"\nSampel pertama:")
print(f"  input_ids      : {sample['input_ids'][:10]} ...")
print(f"  attention_mask : {sample['attention_mask'][:10]} ...")
print(f"  label          : {sample['label']}")

In [ ]:
# Check if tokenization was correct
sample      = train_dataset[0]
decoded     = tokenizer.decode(sample['input_ids'], skip_special_tokens=False)
text_asli   = df_train.iloc[0]['text']
label_asli  = df_train.iloc[0]['label']

print(f"Teks asli   : {text_asli}")
print(f"Teks decoded: {decoded}")
print(f"Label       : {label_asli}")

In [ ]:
import torch.nn as nn
from transformers import AutoModel

class BertSentenceClassifier(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(model_name)
        self.dropout    = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # ambil token [CLS]
        cls_output = self.dropout(cls_output)
        return self.classifier(cls_output)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = BertSentenceClassifier(MODEL_NAME).to(device)

print(f"Device          : {device}")
print(f"Hidden size     : {model.bert.config.hidden_size}")
print(f"Total parameter : {sum(p.numel() for p in model.parameters()):,}")

# **1st Training**

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

print("> RUN 1st TRAINING ...")

# Konstanta
EPOCHS     = 3
LR         = 2e-5
POS_WEIGHT = 5.5

# Class weight
class_weights = torch.tensor([1.0, POS_WEIGHT], dtype=torch.float).to(device)
criterion     = nn.CrossEntropyLoss(weight=class_weights)
optimizer     = AdamW(model.parameters(), lr=LR)
total_steps   = len(train_loader) * EPOCHS
scheduler     = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = int(0.1 * total_steps),
    num_training_steps = total_steps
)


def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.set_grad_enabled(training):
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)

            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                scheduler.step()

            total_loss += loss.item()
            preds       = logits.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

    return total_loss / len(loader), correct / total


best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss,   val_acc   = run_epoch(val_loader,   training=False)

    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"  Train — loss: {train_loss:.4f} | acc: {train_acc:.4f}")
    print(f"  Val   — loss: {val_loss:.4f} | acc: {val_acc:.4f}")

    # Simpan model terbaik berdasarkan val_loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f"{TRAINED_MODEL_PATH}/best_model_v1.pt")
        print(f"Model saved (val_loss: {val_loss:.4f})")

# **1st Evaluation**

In [ ]:
from sklearn.metrics import classification_report
from rouge_score import rouge_scorer

print("> RUN 1st EVAL ...")

# Load model terbaik (epoch 2)
model.load_state_dict(torch.load(f"{TRAINED_MODEL_PATH}/best_model_v1.pt"))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        logits = model(input_ids, attention_mask)
        preds  = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Classification report
print("=== Classification Report ===")
print(classification_report(all_labels, all_preds, target_names=["Not Summary", "Summary"]))

# ROUGE per dokumen
df_test["pred"] = all_preds
scorer          = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
r1, r2, rL      = [], [], []

for doc_id, group in df_test.groupby("doc_id"):
    candidate = " ".join(group[group["pred"]  == 1]["text"].tolist())
    reference = " ".join(group[group["label"] == 1]["text"].tolist())

    if not candidate.strip():
        continue

    scores = scorer.score(reference, candidate)
    r1.append(scores["rouge1"].fmeasure)
    r2.append(scores["rouge2"].fmeasure)
    rL.append(scores["rougeL"].fmeasure)

print("=== ROUGE Score ===")
print(f"  ROUGE-1 : {sum(r1) / len(r1):.4f}")
print(f"  ROUGE-2 : {sum(r2) / len(r2):.4f}")
print(f"  ROUGE-L : {sum(rL) / len(rL):.4f}")
print(f"  (dari {len(r1)} dokumen)")

# 2nd **Training**

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

print("> RUN 2nd TRAINING ...")

POS_WEIGHT    = 2.0
LR            = 1e-5
EPOCHS        = 5
PATIENCE      = 2 # stop kalau val_loss tidak membaik 2 epoch berturut-turut

class_weights = torch.tensor([1.0, POS_WEIGHT], dtype=torch.float).to(device)
criterion     = nn.CrossEntropyLoss(weight=class_weights)
optimizer     = AdamW(model.parameters(), lr=LR)
total_steps   = len(train_loader) * EPOCHS
scheduler     = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = int(0.1 * total_steps),
    num_training_steps = total_steps
)


def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.set_grad_enabled(training):
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)

            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                scheduler.step()

            total_loss += loss.item()
            preds       = logits.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

    return total_loss / len(loader), correct / total


best_val_loss    = float("inf")
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss,   val_acc   = run_epoch(val_loader,   training=False)

    print(f"Epoch {epoch}/{EPOCHS}", flush=True)
    print(f"  Train — loss: {train_loss:.4f} | acc: {train_acc:.4f}", flush=True)
    print(f"  Val   — loss: {val_loss:.4f} | acc: {val_acc:.4f}", flush=True)

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), f"{TRAINED_MODEL_PATH}/best_model_v2.pt")
        print(f"Model disimpan (val_loss: {val_loss:.4f})", flush=True)
    else:
        patience_counter += 1
        print(f"Tidak membaik ({patience_counter}/{PATIENCE})", flush=True)

        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping di epoch {epoch}", flush=True)
            break

# **2nd Eval**

In [ ]:
from sklearn.metrics import classification_report
from rouge_score import rouge_scorer
import torch.nn.functional as F

print("> RUN 2nd EVAL ...")

# Load model terbaik
model.load_state_dict(torch.load(f"{TRAINED_MODEL_PATH}/best_model_v2.pt"))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        logits = model(input_ids, attention_mask)
        preds  = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("=== Classification Report ===", flush=True)
print(classification_report(all_labels, all_preds, target_names=["Not Summary", "Summary"]), flush=True)

# ROUGE
df_test["pred"] = all_preds
scorer          = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)
r1, r2, rL      = [], [], []

for doc_id, group in df_test.groupby("doc_id"):
    candidate = " ".join(group[group["pred"]  == 1]["text"].tolist())
    reference = " ".join(group[group["label"] == 1]["text"].tolist())

    if not candidate.strip():
        continue

    scores = scorer.score(reference, candidate)
    r1.append(scores["rouge1"].fmeasure)
    r2.append(scores["rouge2"].fmeasure)
    rL.append(scores["rougeL"].fmeasure)

print("=== ROUGE Score ===", flush=True)
print(f"  ROUGE-1 : {sum(r1) / len(r1):.4f}", flush=True)
print(f"  ROUGE-2 : {sum(r2) / len(r2):.4f}", flush=True)
print(f"  ROUGE-L : {sum(rL) / len(rL):.4f}", flush=True)
print(f"  (dari {len(r1)} dokumen)")